# Relaxation and Time Evolution

A spin-half can lose population imbalance on different timescales. Use tunneling-induced inelastic transitions for relaxation.

Rates are in inverse seconds $\mathrm{1/s}$ and evolution times in $\mathrm{ns}$.

In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from qutip import Qobj, expect

from spinfinity import SpinSys, bath, hamil, lind, plotting

pio.renderers.default = "plotly_mimetype+notebook"
# Use "fixed" for compact plots, or "no" to suppress display.
plotting.set_display_mode("responsive")

## Hamiltonian and dissipation

The field is along $z$, so the energy and local-z bases coincide. At zero voltage the unpolarized junction is in equilibrium. For this two-level model,

$$
T_1^{-1}=\Gamma_\uparrow+\Gamma_\downarrow
$$

In [ ]:
spin = SpinSys([0.5])
spin.B = [0.0, 0.0, 0.5]
spin.BTip = [0.0, 0.0, 0.0]
spin.gfactorvector = [2.0]
spin.tip = "f"

spin.T = 0.5
spin.V_DC = 0.0
spin.G_st = 1e-10
spin.G_ss = [1e-9]
spin.G_tt = 1
spin.U = np.array([0.0])
spin.b0 = 0.0
spin.TipPolarization = [0.0, 0.0, 0.0]
spin.SamplePolarization = [0.0, 0.0, 0.0]

hamil.calc_EigEnergies(spin, rebuildH=True)
bath.calc_Rates(spin)


relaxation_ops = lind.calc_collapseOperators(spin, recalcRates=False)
spin.collapse_ops = relaxation_ops

rate_up = spin.Rates_Summed[0, 1]  # 1/s
rate_down = spin.Rates_Summed[1, 0]  # 1/s
T1 = 1e9 / (rate_up + rate_down)  # ns
p_equilibrium = bath.calc_ThermalPopulations(spin)[1]

ground = Qobj(spin.eigVectors[:, 0])
excited = Qobj(spin.eigVectors[:, 1])

T1 = 670.37 ns
T_phi = 200.00 ns
T2 = 174.04 ns


## Population relaxation

Starting in the excited state, its population approaches thermal equilibrium.

The time-evolution plot shows the normalized spin expectation value
$2\langle S_z\rangle$.
For this spin-half system, the two are related by
$p_e=(1-2\langle S_z\rangle)/2$.

In [ ]:
spin.psi0 = excited
times_population = np.linspace(0.0, 5.0 * T1, 401)

result_population = lind.calc_TimeEvolution(
    spin, psi_0=spin.psi0, time_array=times_population
)

# The plot function plots the last computed result
lind.plot_TimeEvolution(spin);

